# pt_createverdicts

Loads today's race JSONs (from PT_Vorarbeiten precomputed pickle), generates
AI verdicts via Claude, injects them into the base HTMLs, and saves verdict JSONs
for PT_monitor_learning.

**Requires:** `precomputed_tdy_{TODAY}.pkl` produced by PT_Vorarbeiten.

**Produces:**
- `races/*.html` — HTMLs with AI verdicts injected
- `race_verdicts/{date}_{meeting}.json` — verdict JSONs for monitoring
- `precomputed_tdy_{TODAY}.pkl` — updated with `all_race_verdicts`


In [ ]:
import os, json, time, pickle
from datetime import date

!pip install -q anthropic
import anthropic


In [ ]:
# papermill parameters — injected by PT_WORKFLOW.ipynb when run automatically
ANTHROPIC_API_KEY = None


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
BASE  = '/content/drive/MyDrive/PT' if os.path.exists('/content/drive/MyDrive/PT') else '.'
TODAY = date.today().strftime('%Y-%m-%d')

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')

print(f'BASE:  {BASE}')
print(f'TODAY: {TODAY}')


In [ ]:
# Load system prompts + pt_html_functions from repo
import os as _os, urllib.request as _urlreq
_token = _os.environ.get('GITHUB_TOKEN', '')
_url   = 'https://raw.githubusercontent.com/arauch6363-crypto/ptnew/main/pt_html_functions.py'
_req   = _urlreq.Request(_url, headers={'Authorization': f'token {_token}'} if _token else {})
with _urlreq.urlopen(_req) as _r:
    exec(compile(_r.read(), 'pt_html_functions.py', 'exec'), globals())
print('✅ pt_html_functions loaded')


In [ ]:
# ── Load precomputed from PT_Vorarbeiten ────────────────────────────────────
_pkl_path = f'{BASE}/precomputed_tdy_{TODAY}.pkl'
with open(_pkl_path, 'rb') as _f:
    _precomputed = pickle.load(_f)

race_jsons = _precomputed['race_jsons']
print(f'✅ Loaded precomputed_tdy_{TODAY}.pkl  (race_jsons: {len(race_jsons)})')


## Generate AI Verdicts


In [ ]:
# ── Load learnings DB + Generate AI verdicts per race ──────────────────────
_learnings_path = f'{BASE}/learnings_db.json'
learnings_db = []
if os.path.exists(_learnings_path):
    with open(_learnings_path) as _f:
        learnings_db = json.load(_f)
print(f'📚 Loaded {len(learnings_db)} learnings from learnings_db.json')

all_race_verdicts = {}
_total = len(race_jsons)
_idx   = 0

for race_key, race_json in race_jsons.items():
    _idx += 1
    print(f'[{_idx}/{_total}] {race_json.get("meeting","")} — {race_json.get("race","")}',
          end=' ... ', flush=True)
    try:
        _combined = generate_combined_verdict(race_json, api_key=ANTHROPIC_API_KEY,
                                              learnings_db=learnings_db or None)
        if _combined:
            _rv = {k: _combined[k] for k in ('nap', 'each_way') if k in _combined}
            if _rv.get('nap') and _rv.get('each_way'):
                all_race_verdicts[race_key] = _rv
                _nap = _rv['nap'].get('horse', '?')
                _ew  = _rv['each_way'].get('horse', '?')
                print(f'NAP={_nap} EW={_ew} ✅')
            else:
                print('nap/ew missing ⚠️')
        else:
            print('empty response ⚠️')
    except Exception as _e:
        print(f'ERR: {_e}')
    time.sleep(1)

print(f'\n✅ Race verdicts: {len(all_race_verdicts)}')


## Inject & Save


In [ ]:
# ── Inject verdicts into HTML + save verdict JSONs + update precomputed ─────
BASE_OUT = f'{BASE}/races'

# Inject race-level NAP/EW verdicts into HTMLs
update_race_verdicts_in_html(BASE_OUT, TODAY, all_race_verdicts)

# Save race verdict JSONs: PT/race_verdicts/{date}_{meeting}.json
_rv_dir = f'{BASE}/race_verdicts'
os.makedirs(_rv_dir, exist_ok=True)

_by_meeting = {}
for race_key, race_json in race_jsons.items():
    _meeting = race_json.get('meeting', 'unknown')
    _safe_m  = _meeting.replace(' ', '_').replace('/', '_')
    if _safe_m not in _by_meeting:
        _by_meeting[_safe_m] = []

    _rv = all_race_verdicts.get(race_key, {})
    _nap = dict(_rv.get('nap') or {})
    _ew  = dict(_rv.get('each_way') or {})

    _saddle_map = {h['name']: h.get('saddle') for h in race_json.get('horses', [])}
    if _nap.get('horse'):
        _nap['saddle'] = _saddle_map.get(_nap['horse'])
    if _ew.get('horse'):
        _ew['saddle'] = _saddle_map.get(_ew['horse'])

    _nap.pop('reason', None)
    _ew.pop('reason', None)

    _by_meeting[_safe_m].append({
        'raceId':           race_json.get('raceId'),
        'race':             race_json.get('race'),
        'meeting':          _meeting,
        'going':            race_json.get('going'),
        'going_category':   race_json.get('going_category'),
        'distance_m':       race_json.get('distance_m'),
        'distance_group':   race_json.get('distance_group'),
        'race_type':        race_json.get('race_type'),
        'race_class':       race_json.get('race_class'),
        'total_prize_eur':  race_json.get('total_prize_eur'),
        'field_size':       race_json.get('field_size'),
        'paristurf_verdict': race_json.get('paristurf_verdict'),
        'nap':      _nap,
        'each_way': _ew,
        'horses':   race_json.get('horses', []),
    })

for _safe_m, _races in _by_meeting.items():
    _jpath = f'{_rv_dir}/{TODAY}_{_safe_m}.json'
    with open(_jpath, 'w', encoding='utf-8') as _f:
        json.dump(_races, _f, ensure_ascii=False, indent=2, default=str)
    print(f'  Saved {_jpath} ({len(_races)} races)')

# Update precomputed with all_race_verdicts
_pkl_path = f'{BASE}/precomputed_tdy_{TODAY}.pkl'
with open(_pkl_path, 'rb') as _f:
    _precomputed = pickle.load(_f)
_precomputed['all_race_verdicts'] = all_race_verdicts
with open(_pkl_path, 'wb') as _f:
    pickle.dump(_precomputed, _f)

print(f'\n✅ Precomputed updated with all_race_verdicts ({len(all_race_verdicts)})')
print(f'✅ Verdict JSONs saved to: {_rv_dir}')
print(f'✅ HTMLs with AI verdicts: {BASE_OUT}')


In [ ]:
print('\n✅ pt_createverdicts complete.')
print(f'   Races processed:  {len(race_jsons)}')
print(f'   Verdicts generated: {len(all_race_verdicts)}')
print(f'   race_verdicts/: {TODAY}_*.json')
print('\nNow run PT_Create_HTMLs_fast.ipynb ▶')
